### Training a CNN to predict sleep stages from EEG Signals.

In [ ]:
import os 
import glob 

import numpy as np
import matplotlib.pyplot as plt 

import torch 
import torch.nn as nn 

from sklearn.model_selection import train_test_split 
from sklearn.metrics import accuracy_score, confusion_matrix

import mne # EEG package (ie. reading edf & hypnogram files)

In [ ]:
data_dir = "data/sleep_data/sleep-cassette"

#store the filepath of specfic files join (___, this) as a list. 
psg_files = sorted(glob.glob(os.path.join(data_dir, '*PSG.edf')))
hypnogram_files = sorted(glob.glob(os.path.join(data_dir, '*Hypnogram.edf')))

#### Data Exploration

In [ ]:
# loading oen of the raw time-series psg data
raw_edf = mne.io.read_raw_edf(psg_files[0], preload=True, verbose=False)

In [ ]:
raw_edf 

#7 channels, only 2 for EEG

'''
ch1 EEG Fpz-Cz
ch2 EEG Pz-Oz
ch3 EOG Horizontal # Tracking eye movement
ch4 Resp oro-nasal # Measurments of air movement through the nose and mouth
ch5 EMG submental # Tracking electrical signal under chin 
ch6 Temp rectal # Temprature of body
ch7 Event marker #inserting timestamped labels into eeg recordign when 
                # there is something meaningful to mark 
'''

In [ ]:
# getting actual eeg data from th edf file, use .get_data()
eeg_data = raw_edf.get_data()

In [ ]:
eeg_data

In [ ]:
# getting eeg of only first channel 

eeg_channel1 = eeg_data[0]

# 0 = 1st channel, 6 = 7th channel

In [ ]:
plt.plot(eeg_channel1[:2000])

In [ ]:
#plotting all the data to see the different channels that we have
fig, axs = plt.subplots(nrows=7, ncols=1, figsize=(12, 8))

for i, ax in enumerate(axs):
    ax.plot(eeg_data[i][:1000])

plt.tight_layout()

In [ ]:
# proper visualization wiht proper labeling for the 2 eeg channels 
eeg_channels = ['EEG Fpz-Cz', 'EEG Pz-Oz']

eeg_data = raw_edf.get_data(picks=eeg_channels)

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(14, 6))


fs = 100 # Sampling frequency
N = 3000 # for 30s of data 

for i, ax in enumerate(axs):
    x = np.arange(N) / fs # -> 0.01s

    ax.plot(x, eeg_data[i][:N])
    ax.set_title(eeg_channels[i])
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Voltage (uV)')

plt.tight_layout

Ground truth (hypnogram files)

In [ ]:
annotations = mne.read_annotations(hypnogram_files[0])

In [ ]:
annotations.onset

In [ ]:
annotations.duration

In [ ]:
annotations.description

In [ ]:
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(14,6))

fs = 100
N = 7950000

# Axis 1 - EEG
x = np.arange(N) / fs

axs[0].plot(x, eeg_data[0][:N])
axs[0].set_title(eeg_channels[0])
axs[0].set_xlabel('Seconds (s)')
axs[0].set_ylabel('Voltage (uV)')

axs[0].set_xlim(0, N/fs)

# Axis 2 - EEG
x = np.arange(N) / fs

axs[1].plot(x, eeg_data[1][:N])
axs[1].set_title(eeg_channels[1])
axs[1].set_xlabel('Seconds (s)')
axs[1].set_ylabel('Voltage (uV)')

axs[1].set_xlim(0, N/fs)

# Axis 3 - Hypnogram
t_end = N / fs

stage_map = {'Sleep stage W': 0, 'Sleep stage 1': 1, 'Sleep stage 2': 2, 'Sleep stage 3': 3, 'Sleep stage 4': 4, 'Sleep stage R': 5}

x, y = [], []
for o, d, s in zip(annotations.onset, annotations.duration, annotations.description):
    start, end = max(o, 0), min(o + d, t_end)
    stage = stage_map.get(s, np.nan)
    x += [start, end]
    y += [stage, stage]

axs[2].step(x, y, where='post')
axs[2].set_xlim(0, t_end)
axs[2].set_ylim(-0.5, 5.5)
axs[2].set_yticks([0,1,2,3,4,5])
axs[2].set_yticklabels(['W','N1','N2','N3', 'N4', 'R'])
axs[2].set_xlabel('Seconds (s)')
axs[2].set_title('Hypnogram')

plt.tight_layout()

### Data Preperation

For now, only 1 channel is used to make the predictions. (im lazy!)

In [ ]:
len(psg_files)

# 153!! too much it will blow up the ram!, just 25 samples of the data will be used. 

In [ ]:
X_data = []
y_data = []
window_length = 3000 # 30 seconds length of video data, each time, to determine sleep  


for psg_file, hypnogram_file in zip(psg_files[:25], hypnogram_files[:25]):
    raw_edf = mne.io.read_raw_edf(psg_file, preload=True, verbose=False)
    
    eeg_channel1 = raw_edf.get_data(picks=['EEG Fpz-Cz'])[0] #unknown data, to be trained
    annotations = mne.read_annotations(hypnogram_file) # known data

    stage_map = {'Sleep stage W': 0, 
                 'Sleep stage 1': 1, 
                 'Sleep stage 2': 2, 
                 'Sleep stage 3': 3, 
                 'Sleep stage 4': 4, 
                 'Sleep stage R': 5}

    for i, (onset, duration, description) in enumerate(zip(annotations.onset, annotations.duration, annotations.description)):
        if description not in stage_map:
            continue

        for segment in range(int(onset * fs ), int(onset * fs  + duration * fs), window_length):
            start_sample = segment
            end_sample = segment + window_length

            if end_sample <= len(eeg_channel1):
                window = eeg_channel1[start_sample:end_sample]

                if len(window) == window_length:
                    X_data.append(window)
                    y_data.append(stage_map[description])

X = np.array(X_data)
y = np.array(y_data)
            


                                                       
    

Analyzing data:

In [ ]:
X.shape

In [ ]:
y.shape

In [ ]:
np.bincount(y)

2 Steps to use the data:
1. Normalize it 
2. train-test split with scikit-learn

In [ ]:
X = (X - X.mean()) / X.std()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
# 20% data used for training, and 80% used for trianing. 

In [ ]:
np.bincount(y_train) / len(y_train)

In [ ]:
np.bincount(y_test) / len(y_test)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train = torch.FloatTensor(X_train).unsqueeze(1).to(device)
X_test = torch.FloatTensor(X_test).unsqueeze(1).to(device)
y_train = torch.LongTensor(y_train).to(device)
y_test = torch.LongTensor(y_test).to(device)

### Model Training (CNN)

In [ ]:
model = nn.Sequential(
    nn.Conv1d(1, 32, kernel_size=50, stride=6),
    nn.ReLU(),
    nn.MaxPool1d(8),

    nn.Conv1d(32, 64, kernel_size=8),
    nn.ReLU(),
    nn.MaxPool1d(8),
    nn.Flatten(),
    nn.Linear(64 * 6, 128),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(128, 6)
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 10
batch_size = 64 

for epoch in range(epochs):
    model.train()

    total_loss = 0 

    for i in range(0, len(X_train), batch_size):
        batch_X = X_train[i:i+batch_size].to(device)
        batch_y = y_train[i:i+batch_size].to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)
        loss.backward()

        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(X_train):.4f}")

In [ ]:
model.eval()

with torch.no_grad():
    predictions = model(X_test)
    predicted_classes = torch.argmax(predictions, dim=1).cpu().numpy()

accuracy = accuracy_score(y_test.cpu().numpy(), predicted_classes)

In [ ]:
accuracy

In [ ]:
confusion_matrix(y_test.cpu().numpy(), predicted_classes)